# Turning Treasury Yields Into Supervised Learning Windows

Phase 1 gave us clean daily Treasury yields. Phase 2 helped us look at the data like analysts. Phase 3 turns the time series into supervised-learning examples that a sequence model can eventually consume.

We still are **not training a model** here. We are defining the learning problem carefully.

## Why this is supervised learning

A raw time series is just observations ordered by date. Supervised learning needs examples with inputs and answers.

For each forecast origin date `t`:

- `X` is the information available up to `t`.
- `y` is what happens after `t`.

That pairing is what makes the dataset supervised. The model will later learn a mapping from recent yield-curve history to future yield changes.

## The concrete example

With `lookback = 60` and `horizon = 1`:

- `X` is the previous 60 trading days of features ending at date `t`.
- `y` is the next trading day's seven yield changes, from `t` to `t + 1`.

For each maturity we create four features: level, 1-day change, 5-day change, and 21-day change. There are seven maturities, so each day has `7 * 4 = 28` features.

Final tensor shapes:

- one input example: `(60, 28)`
- one target example: `(7,)`
- full input tensor: `(num_examples, 60, 28)`
- full target tensor: `(num_examples, 7)`

In [1]:
# ruff: noqa: E402, I001
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from market_resonance.features import (
    build_supervised_windows,
    chronological_train_validation_test_split,
    create_treasury_features,
    standardize_splits,
)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "treasury_yields_daily.csv"

In [2]:
yields = pd.read_csv(DATA_PATH, parse_dates=["date"])
yields.head()

,date,3M,6M,1Y,2Y,5Y,10Y,30Y
0,1990-01-02,7.83,7.89,7.81,7.87,7.87,7.94,8.00
1,1990-01-03,7.89,7.94,7.85,7.94,7.92,7.99,8.04
2,1990-01-04,7.84,7.90,7.82,7.92,7.91,7.98,8.04
3,1990-01-05,7.79,7.85,7.79,7.90,7.92,7.99,8.06
4,1990-01-08,7.79,7.88,7.81,7.90,7.92,8.02,8.09


## Feature table

The feature table stays aligned by date. The early rows have missing trailing-change values because a 21-day change cannot exist until we have at least 21 earlier trading observations. The window builder starts only once all required features are available.

In [3]:
features = create_treasury_features(yields)
features.filter(regex="date|10Y").head(25).tail()

,date,10Y_level,10Y_chg_1d,10Y_chg_5d,10Y_chg_21d
20,1990-01-31,8.43,-0.08,0.05,NaN
21,1990-02-01,8.42,-0.01,0.00,0.48
22,1990-02-02,8.50,0.08,0.01,0.51
23,1990-02-05,8.53,0.03,0.03,0.55
24,1990-02-06,8.57,0.04,0.06,0.58


## Build supervised windows

The reusable Python module creates batch-first arrays. The feature tensor is ready for sequence models such as an LSTM, but no model is trained in this phase.

In [4]:
windows = build_supervised_windows(yields, lookback=60, horizon=1)
{
    "X_shape": windows.X.shape,
    "y_shape": windows.y.shape,
    "num_features": len(windows.feature_columns),
    "num_targets": len(windows.target_columns),
}

{'X_shape': (9096, 60, 28),
 'y_shape': (9096, 7),
 'num_features': 28,
 'num_targets': 7}

## Inspect one `X` and `y` pair

The first example below has 60 rows of input history. Its target date is the next trading row after the input window ends.

In [5]:
example_index = 0
example_X = windows.X[example_index]
example_y = windows.y[example_index]

{
    "sample_start": windows.sample_start_dates.iloc[example_index].date(),
    "sample_end": windows.sample_end_dates.iloc[example_index].date(),
    "target_date": windows.target_dates.iloc[example_index].date(),
    "X_shape": example_X.shape,
    "y_shape": example_y.shape,
}

{'sample_start': datetime.date(1990, 2, 1),
 'sample_end': datetime.date(1990, 4, 27),
 'target_date': datetime.date(1990, 4, 30),
 'X_shape': (60, 28),
 'y_shape': (7,)}

In [6]:
pd.DataFrame(example_X, columns=windows.feature_columns).tail()

,3M_level,3M_chg_1d,3M_chg_5d,3M_chg_21d,6M_level,6M_chg_1d,6M_chg_5d,6M_chg_21d,1Y_level,1Y_chg_1d,...,5Y_chg_5d,5Y_chg_21d,10Y_level,10Y_chg_1d,10Y_chg_5d,10Y_chg_21d,30Y_level,30Y_chg_1d,30Y_chg_5d,30Y_chg_21d
55,8.00,0.05,-0.01,-0.17,8.31,0.07,0.13,0.04,8.50,0.06,...,0.27,0.36,8.98,0.03,0.30,0.45,8.96,0.03,0.32,0.47
56,8.03,0.03,0.00,-0.12,8.36,0.05,0.13,0.10,8.55,0.05,...,0.23,0.42,9.00,0.02,0.23,0.48,8.98,0.02,0.24,0.50
57,8.06,0.03,-0.02,-0.07,8.40,0.04,0.10,0.12,8.57,0.02,...,0.16,0.48,9.01,0.01,0.15,0.50,8.98,0.00,0.13,0.51
58,8.09,0.03,0.03,-0.08,8.45,0.05,0.17,0.15,8.64,0.07,...,0.25,0.54,9.07,0.06,0.20,0.55,9.04,0.06,0.18,0.56
59,8.05,-0.04,0.10,-0.07,8.38,-0.07,0.14,0.11,8.57,-0.07,...,0.18,0.53,9.06,-0.01,0.11,0.55,9.04,0.00,0.11,0.57


In [7]:
pd.Series(example_y, index=windows.target_columns)

3M_target_chg_1d     0.02
6M_target_chg_1d     0.06
1Y_target_chg_1d     0.01
2Y_target_chg_1d    -0.02
5Y_target_chg_1d    -0.02
10Y_target_chg_1d   -0.02
30Y_target_chg_1d   -0.04
dtype: float64

## Chronological train/validation/test split

Time series examples must stay in chronological order. Random shuffling would mix future regimes into training and make validation or test results too optimistic.

In [8]:
splits = chronological_train_validation_test_split(windows)
{
    "train": splits.train.X.shape,
    "validation": splits.validation.X.shape,
    "test": splits.test.X.shape,
    "train_end": splits.train.sample_end_dates.iloc[-1].date(),
    "validation_end": splits.validation.sample_end_dates.iloc[-1].date(),
    "test_end": splits.test.sample_end_dates.iloc[-1].date(),
}

{'train': (6367, 60, 28),
 'validation': (1364, 60, 28),
 'test': (1365, 60, 28),
 'train_end': datetime.date(2015, 10, 5),
 'validation_end': datetime.date(2021, 3, 23),
 'test_end': datetime.date(2026, 9, 4)}

## Normalize without leakage

Normalization is fitted on `train.X` only. The frozen mean and scale are then applied to validation and test. This prevents validation/test distribution information from leaking backward into training.

In [9]:
standardized_splits, standardizer = standardize_splits(splits)
{
    "standardized_train_X_shape": standardized_splits.train.X.shape,
    "mean_shape": standardizer.mean_.shape,
    "scale_shape": standardizer.scale_.shape,
    "train_mean_after_standardization": standardized_splits.train.X.mean().round(6),
    "train_std_after_standardization": standardized_splits.train.X.std().round(6),
}

{'standardized_train_X_shape': (6367, 60, 28),
 'mean_shape': (1, 1, 28),
 'scale_shape': (1, 1, 28),
 'train_mean_after_standardization': np.float64(0.0),
 'train_std_after_standardization': np.float64(1.0)}

## 02 Stopping point

At this point we have supervised arrays, chronological splits, and train-only normalization. That is the data interface a model can use later. 